# Visualizzatore delle metriche

Ordina le immagini secondo una metrica (dal valore piu' basso al piu' alto) e mostra i ritagli
del volto inpaintato salvati da `prepare.py`, con il contorno della maschera sovrapposto.

- **quantili**: N immagini distribuite uniformemente lungo la classifica (minimo, ..., massimo):
  si vede subito cosa cambia al crescere della metrica.
- **pagina**: N immagini consecutive a partire dalla posizione `start` (per scorrere la classifica).

Sopra la griglia, l'istogramma della metrica con i valori delle immagini mostrate.
Legge `index.csv` + `a_coherence.csv` / `b_quality.csv` / `c_prompt.csv` (quelli presenti) dalla cartella `OUT`.

In [ ]:
from pathlib import Path
import sys

SCRIPTS = Path(".")          # cartella con common.py
#OUT = Path("/Scratch/Shullani/Dataset/results")         # cartella --out degli script
OUT = Path("/var/home/da/Downloads/results-lamarr")

sys.path.insert(0, str(SCRIPTS.resolve()))
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from common import load_merged, load_mask, extract

df = load_merged(OUT)

# colonne che non sono metriche (geometria, contatori, configurazione)
NOT_METRIC = {"width", "height", "mask_invert", "n_faces", "face_found",
              *[f"{p}_{k}" for p in ("face", "mask", "crop") for k in ("x0", "y0", "x1", "y1")]}
METRICS = [c for c in df.select_dtypes("number").columns if c not in NOT_METRIC and df[c].notna().any()]
# metriche dove PIU' BASSO = MEGLIO (solo per ricordarlo nel titolo)
LOWER_IS_BETTER = {"brisque", "niqe"}

print(f"{len(df)} immagini, crop_kind: {df['crop_kind'].value_counts().to_dict()}")
print("metriche disponibili:", METRICS)

## Funzioni

In [ ]:
def crop_with_mask(row, overlay=True, color=(255, 64, 64)):
    """Ritaglio salvato (RGB) con il contorno della maschera proiettato sopra."""
    crop = cv2.imread(str(OUT / row["crop_file"]), cv2.IMREAD_COLOR)
    if crop is None:
        return None
    crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    mp = row.get("mask_path")
    if overlay and isinstance(mp, str) and mp and pd.notna(row.get("crop_x0")):
        shape = (int(row["height"]), int(row["width"]))
        m = load_mask(mp, shape, bool(row.get("mask_invert", 0)))
        if m is not None:
            box = tuple(int(row[f"crop_{k}"]) for k in ("x0", "y0", "x1", "y1"))
            mc = extract(m.astype(np.uint8) * 255, box, (crop.shape[1], crop.shape[0]),
                         border=cv2.BORDER_CONSTANT, interp=cv2.INTER_NEAREST)
            cnts, _ = cv2.findContours((mc > 127).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            cv2.drawContours(crop, cnts, -1, color, max(1, crop.shape[0] // 256))
    return crop


def select(metric, n=24, mode="quantili", start=0, kind="face", flag="", descending=False):
    """Righe ordinate per metrica (crescente di default) e scelte secondo mode."""
    d = df[df[metric].notna()]
    if kind and kind != "tutti":
        d = d[d["crop_kind"] == kind]
    if flag:
        d = d[d.get("flags", pd.Series("", index=d.index)).fillna("").str.contains(flag)]
    d = d.sort_values(metric, ascending=not descending, kind="mergesort").reset_index(drop=True)
    d["rank"] = np.arange(len(d))
    d["pct"] = 100 * d["rank"] / max(len(d) - 1, 1)
    if len(d) == 0:
        return d
    if mode == "quantili":
        pos = np.unique(np.round(np.linspace(0, len(d) - 1, min(n, len(d)))).astype(int))
        return d.iloc[pos]
    return d.iloc[start:start + n]


def show(metric, n=24, mode="quantili", start=0, kind="face", flag="", descending=False,
         overlay=True, cols=6, tile=2.6):
    sel = select(metric, n, mode, start, kind, flag, descending)
    base = df[df[metric].notna()]
    if kind and kind != "tutti":
        base = base[base["crop_kind"] == kind]
    if sel.empty:
        print(f"Nessuna immagine con {metric} (crop_kind={kind}, flag='{flag}')")
        return
    rows = int(np.ceil(len(sel) / cols))
    fig = plt.figure(figsize=(cols * tile, 1.6 + rows * (tile + 0.5)))
    gs = gridspec.GridSpec(rows + 1, cols, height_ratios=[1.6 / tile] + [1] * rows, hspace=0.45, wspace=0.05)

    # istogramma con i valori mostrati
    ax = fig.add_subplot(gs[0, :])
    ax.hist(base[metric], bins=60, color="#9aa5b1")
    for v in sel[metric]:
        ax.axvline(v, color="#d1495b", lw=0.8, alpha=0.8)
    better = "piu' basso = meglio" if metric in LOWER_IS_BETTER else ""
    order = "decrescente" if descending else "crescente"
    ax.set_title(f"{metric}  ({len(base)} immagini, ordine {order}{', ' + better if better else ''})", fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_yticks([])

    for i, (_, r) in enumerate(sel.iterrows()):
        a = fig.add_subplot(gs[1 + i // cols, i % cols])
        img = crop_with_mask(r, overlay)
        if img is not None:
            a.imshow(img)
        flags = r.get("flags") if isinstance(r.get("flags"), str) else ""
        title = f"{r[metric]:.4g}  ·  p{r['pct']:.0f}\n{Path(r['file']).stem.split('_inpainted')[0][:28]}"
        if flags:
            title += f"\n{flags}"
        a.set_title(title, fontsize=7.5, color="#b00020" if flags else "black")
        a.set_xticks([]); a.set_yticks([])
    plt.show()


def show_frame(file, overlay=True):
    """Frame intero con maschera (rosso), volto rilevato (verde) e box del ritaglio (giallo)."""
    r = df[df["file"] == file].iloc[0]
    img = cv2.cvtColor(cv2.imread(str(r["image_path"])), cv2.COLOR_BGR2RGB)
    lw = max(1, img.shape[0] // 300)
    if overlay and isinstance(r.get("mask_path"), str) and r["mask_path"]:
        m = load_mask(r["mask_path"], img.shape, bool(r.get("mask_invert", 0)))
        cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(img, cnts, -1, (255, 64, 64), lw)
    for p, c in (("face", (64, 220, 64)), ("crop", (240, 200, 0))):
        if pd.notna(r.get(f"{p}_x0")):
            x0, y0, x1, y1 = (int(r[f"{p}_{k}"]) for k in ("x0", "y0", "x1", "y1"))
            cv2.rectangle(img, (x0, y0), (x1, y1), c, lw)
    plt.figure(figsize=(10, 10 * img.shape[0] / img.shape[1]))
    plt.imshow(img); plt.axis("off"); plt.title(file, fontsize=9)
    plt.show()
    cols = [c for c in METRICS + ["flags", "prompt"] if c in r.index and pd.notna(r[c])]
    return r[cols].to_frame("valore")

In [ ]:
METRICS

## Viste statiche

In [ ]:
# al crescere della metrica: 24 immagini dal minimo al massimo
metric = "color_dev_frac_in"#next((m for m in ("topiq_nr-face", "musiq", "blur_logratio") if m in METRICS), METRICS[0])
show(metric, n=24, mode="quantili", kind="tutti")

In [ ]:
for i_metric in ['topiq_nr-face', 'musiq', 'clipiqa', 'brisque', 'niqe', 'maniqa', 'clip_score', 'mps_score']:
    show(i_metric, n=24, mode="quantili", kind="tutti")

In [ ]:
# le 6 peggiori secondo la metrica (le prime della classifica crescente)
for i_metric in ['topiq_nr-face', 'musiq', 'clipiqa', 'brisque', 'niqe', 'maniqa', 'clip_score', 'mps_score']:
    show(i_metric, n=6, mode="pagina", start=0, kind="tutti")

In [ ]:
# un singolo frame: maschera, volto rilevato, box del ritaglio e tutte le metriche
show_frame(select(metric, kind="tutti").iloc[0]["file"])